# Neural Network Uncertainty: Three Approaches

**CE 315 — Review Before Michigan's Final Lecture**

---

## Where We Are

We've established the problem:
- Standard neural networks give **point predictions** with no uncertainty
- They're **confidently wrong** in regions with no training data
- Gaussian Processes give great uncertainty but **don't scale** to large problems

Michigan introduced three approaches that give neural networks the ability to express uncertainty. Today we'll walk through each one with code so you understand the mechanics before their final lecture.

All three approaches use the same strategy at a high level: **make multiple predictions and use the spread as uncertainty.** They differ in *how* they generate those multiple predictions.

| Approach | How It Gets Multiple Predictions |
|---|---|
| **MC Dropout** | One network, run many times with random neurons turned off |
| **Mean-Variance Network** | One network that directly outputs μ and σ (not just a point) |
| **Deep Ensemble** | M separate networks, each predicting μ and σ |

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("Ready!")

## Our Test Problem: Noisy Sine With a Gap

This is the same setup Majdi used — a sine wave with noise, and deliberately sparse regions where the model should be uncertain.

In [ ]:
# ============================================================
# Create the test dataset
# ============================================================
np.random.seed(42)

# Dense data on the left, sparse in the middle, dense on the right
X_left = np.random.uniform(0, 3, 60)
X_mid = np.random.uniform(3, 7, 8)       # sparse!
X_right = np.random.uniform(7, 10, 60)
X_train = np.sort(np.concatenate([X_left, X_mid, X_right])).reshape(-1, 1)
y_train = np.sin(X_train.ravel()) + 0.3 * np.random.randn(len(X_train))

# Prediction grid
X_plot = np.linspace(-1, 12, 300).reshape(-1, 1)
y_true = np.sin(X_plot.ravel())

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(X_train, y_train, c='black', s=25, alpha=0.7, label='Training data')
ax.plot(X_plot, y_true, 'k--', alpha=0.3, label='True function')
ax.axvspan(3, 7, alpha=0.08, color='red', label='Sparse region')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Our Test Problem: Where Should the Model Be Uncertain?', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print("Dense data on the left and right, sparse in the middle.")
print("A good UQ method should show HIGH uncertainty in the sparse region")
print("and LOW uncertainty where we have lots of data.")

---

## Approach 1: MC Dropout

### The Idea

You already know **dropout** from the neural network notebook — during training, we randomly "turn off" neurons (set their outputs to zero) to prevent overfitting. Normally, dropout is turned off at prediction time so you get a clean, deterministic output.

**MC Dropout** flips this: keep dropout ON at prediction time. Now, every time you predict the same input, different neurons get turned off, so you get a slightly different answer. Predict 100 times → get 100 answers → the spread of those answers is your uncertainty.

```
Standard NN prediction:    input → network → 4.2 (always the same)

MC Dropout prediction:     input → network (dropout run 1) → 4.1
                           input → network (dropout run 2) → 4.5
                           input → network (dropout run 3) → 3.9
                           input → network (dropout run 4) → 4.3
                           ...100 times...
                           Mean: 4.2, Std: 0.3 ← that's your uncertainty
```

The elegant part: you train the network normally (with dropout for regularization as usual). The uncertainty comes for free at prediction time — no changes to training at all.

In [ ]:
# ============================================================
# MC Dropout: build and train a simple network
# ============================================================
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

# We'll simulate MC Dropout manually since sklearn doesn't expose dropout
# at prediction time. The concept is identical.

# Train a network with a moderate amount of capacity
scaler_X = StandardScaler()
X_train_sc = scaler_X.fit_transform(X_train)
X_plot_sc = scaler_X.transform(X_plot)

# Simulate MC Dropout by training multiple networks with different random
# subsets of neurons active (different random seeds + slight noise injection)
# This approximates what happens when dropout is left on at test time.
n_forward_passes = 100
predictions = []

print(f"Running {n_forward_passes} stochastic forward passes...")
for i in range(n_forward_passes):
    # Each pass uses a different random state, simulating different dropout masks
    nn = MLPRegressor(hidden_layer_sizes=(64, 64), activation='relu',
                       max_iter=500, random_state=i,
                       alpha=0.01)  # some regularization
    nn.fit(X_train_sc, y_train)
    pred = nn.predict(X_plot_sc)
    predictions.append(pred)

predictions = np.array(predictions)  # shape: (100, 300)
mc_mean = predictions.mean(axis=0)
mc_std = predictions.std(axis=0)

print("Done!")

In [ ]:
# ============================================================
# Visualize MC Dropout results
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: spaghetti plot (individual forward passes)
ax = axes[0]
for i in range(min(50, n_forward_passes)):
    ax.plot(X_plot.ravel(), predictions[i], alpha=0.08, color='steelblue', linewidth=1)
ax.scatter(X_train, y_train, c='black', s=20, zorder=5)
ax.plot(X_plot, y_true, 'k--', alpha=0.3)
ax.set_title('MC Dropout: 50 Forward Passes (Spaghetti Plot)', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_ylim(-3, 3)

# Right: mean ± std bands
ax = axes[1]
ax.fill_between(X_plot.ravel(), mc_mean - 2*mc_std, mc_mean + 2*mc_std,
                alpha=0.15, color='steelblue', label='±2σ (95% CI)')
ax.fill_between(X_plot.ravel(), mc_mean - mc_std, mc_mean + mc_std,
                alpha=0.25, color='steelblue', label='±1σ (68% CI)')
ax.plot(X_plot, mc_mean, 'steelblue', linewidth=2.5, label='Mean prediction')
ax.plot(X_plot, y_true, 'k--', alpha=0.3, label='True function')
ax.scatter(X_train, y_train, c='black', s=20, zorder=5)
ax.set_title('MC Dropout: Mean ± Uncertainty Bands', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_ylim(-3, 3)
ax.legend(fontsize=10)

plt.suptitle('Approach 1: MC Dropout', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left: each line is one forward pass with different dropout. They agree")
print("near the data but diverge in sparse regions — that divergence IS uncertainty.")
print("\nRight: summarized as mean ± std bands. Compare to the GP from our UQ notebook —")
print("same shape! Wider in the gap, narrower near data.")

### MC Dropout Summary

| Pros | Cons |
|---|---|
| Dead simple — just leave dropout on at test time | Need many forward passes (slow at prediction time) |
| No changes to training | Uncertainty quality depends heavily on dropout rate |
| Works with any dropout network you already have | Tends to underestimate uncertainty |

---

## Approach 2: Mean-Variance Network

### The Idea

Instead of predicting a single number, make the network output **two numbers**: a mean (μ) and a variance (σ²). The network itself learns where it should be uncertain.

```
Standard NN:          input → network → ŷ = 4.2

Mean-Variance NN:     input → network → μ = 4.2, σ² = 0.09
                                         ↑            ↑
                                     best guess    uncertainty
```

The trick is in the **loss function**. Instead of MSE (which just penalizes wrong predictions), you use **negative log-likelihood (NLL)**:

$$\mathcal{L} = \frac{1}{2}\log(\sigma^2) + \frac{(y - \mu)^2}{2\sigma^2}$$

This loss has two competing terms:
- The second term $(y - \mu)^2 / 2\sigma^2$ says: "make σ large to reduce this penalty" (larger σ = smaller penalty for wrong predictions)
- The first term $\log(\sigma^2)$ says: "make σ small" (penalty for being uncertain)

The network learns the optimal balance: predict small σ where it can be accurate, predict large σ where the data is noisy or sparse.

In [ ]:
# ============================================================
# Mean-Variance Network: build from scratch with numpy
# ============================================================
# We'll implement a simple 2-output network manually to show
# exactly what's happening. In practice you'd use TensorFlow/PyTorch.

# For this demo, we'll train many small networks that each predict
# a point, then fit a local variance estimate. This approximates
# what a mean-variance network learns.

# A cleaner demonstration: train one network for the mean,
# then estimate variance from the residuals in local neighborhoods.

from sklearn.neighbors import KNeighborsRegressor

# Step 1: Train a good network for the mean prediction
nn_mean = MLPRegressor(hidden_layer_sizes=(64, 64), activation='relu',
                        max_iter=1000, random_state=42, alpha=0.001)
nn_mean.fit(X_train_sc, y_train)
mu_pred = nn_mean.predict(X_plot_sc)

# Step 2: Compute squared residuals on training data
train_residuals = (y_train - nn_mean.predict(X_train_sc))**2

# Step 3: A second model learns to predict the variance
# (In a real mean-variance network, both are learned jointly)
nn_var = KNeighborsRegressor(n_neighbors=10, weights='distance')
nn_var.fit(X_train_sc, train_residuals)
sigma2_pred = np.maximum(nn_var.predict(X_plot_sc), 0.01)  # floor at small positive value
sigma_pred = np.sqrt(sigma2_pred)

print("Mean-variance network trained.")
print(f"  Predicted σ range: {sigma_pred.min():.3f} to {sigma_pred.max():.3f}")

In [ ]:
# ============================================================
# Visualize Mean-Variance Network results
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: the two outputs
ax = axes[0]
ax.plot(X_plot.ravel(), mu_pred, '#e74c3c', linewidth=2.5, label='μ (mean prediction)')
ax.plot(X_plot.ravel(), sigma_pred, '#f39c12', linewidth=2.5, label='σ (predicted uncertainty)')
ax.scatter(X_train, y_train, c='black', s=20, zorder=5, alpha=0.5)
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.3)
ax.set_title('Network Outputs: Mean and Uncertainty', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('Value')
ax.legend(fontsize=10)

# Right: confidence bands
ax = axes[1]
ax.fill_between(X_plot.ravel(), mu_pred - 2*sigma_pred, mu_pred + 2*sigma_pred,
                alpha=0.15, color='#e74c3c', label='±2σ')
ax.fill_between(X_plot.ravel(), mu_pred - sigma_pred, mu_pred + sigma_pred,
                alpha=0.25, color='#e74c3c', label='±1σ')
ax.plot(X_plot, mu_pred, '#e74c3c', linewidth=2.5, label='Mean prediction')
ax.plot(X_plot, y_true, 'k--', alpha=0.3, label='True function')
ax.scatter(X_train, y_train, c='black', s=20, zorder=5)
ax.set_title('Mean-Variance Network: Learned Uncertainty Bands', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_ylim(-3, 3)
ax.legend(fontsize=10)

plt.suptitle('Approach 2: Mean-Variance Network', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left: the network outputs TWO values — a mean and a standard deviation.")
print("\nRight: these combine into confidence bands, just like the GP.")
print("The network LEARNED where to be uncertain from the data itself.")

### The NLL Loss: Why It Works

With standard MSE loss, the network has no way to say "I'm not sure about this one" — it pays the same penalty whether the data point is easy or hard.

With NLL loss, the network can "hedge its bets":
- For a data point it can predict well: small σ → small $\log(\sigma^2)$ penalty, small $(y-\mu)^2/\sigma^2$ penalty
- For a data point it can't predict well: it can increase σ to reduce the $(y-\mu)^2/\sigma^2$ term, at the cost of a larger $\log(\sigma^2)$ penalty

The network learns the optimal trade-off: be confident where you're accurate, be uncertain where you're not.

### Mean-Variance Summary

| Pros | Cons |
|---|---|
| Single forward pass (fast at prediction time) | Requires a special loss function |
| Network learns heteroscedastic uncertainty (varying noise) | Can underestimate uncertainty in gaps (no data → no signal) |
| Clean probabilistic interpretation | Needs careful training to avoid σ collapsing to zero |

---

## Approach 3: Deep Ensembles

### The Idea

Combine both ideas above: train **M separate neural networks** (typically 5-10), where each one is a mean-variance network. The total uncertainty comes from two sources:

1. **Aleatoric uncertainty** (data noise): the σ that each individual network predicts
2. **Epistemic uncertainty** (model ignorance): how much the M networks disagree with each other

```
Network 1: μ₁ = 4.1, σ₁ = 0.2
Network 2: μ₂ = 4.4, σ₂ = 0.3
Network 3: μ₃ = 3.9, σ₃ = 0.2
Network 4: μ₄ = 4.3, σ₄ = 0.25
Network 5: μ₅ = 4.2, σ₅ = 0.2
                              
Final: μ = average of μ's = 4.18
       σ² = average of σ²'s + variance of μ's
            (data noise)     (model disagreement)
```

This captures uncertainty that the mean-variance network misses: in a data gap, each individual network might be confidently wrong (small σ) but they'll disagree with each other (high variance of μ's).

In [ ]:
# ============================================================
# Deep Ensemble: train M independent networks
# ============================================================
M = 10  # number of ensemble members

ensemble_means = []

print(f"Training {M} independent networks...")
for m in range(M):
    # Each network gets a different random initialization
    # and sees the data in a different order
    nn_m = MLPRegressor(hidden_layer_sizes=(64, 64), activation='relu',
                         max_iter=1000, random_state=m*7 + 13,
                         alpha=0.001)
    nn_m.fit(X_train_sc, y_train)
    pred_m = nn_m.predict(X_plot_sc)
    ensemble_means.append(pred_m)
    print(f"  Network {m+1}/{M} trained.")

ensemble_means = np.array(ensemble_means)  # shape: (M, 300)

# Ensemble prediction: mean and std across the M networks
ens_mu = ensemble_means.mean(axis=0)
ens_std = ensemble_means.std(axis=0)

print(f"\nEnsemble uncertainty range: {ens_std.min():.4f} to {ens_std.max():.4f}")

In [ ]:
# ============================================================
# Visualize Deep Ensemble results
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: individual ensemble members (spaghetti plot)
ax = axes[0]
colors = plt.cm.tab10(np.linspace(0, 1, M))
for m in range(M):
    ax.plot(X_plot.ravel(), ensemble_means[m], alpha=0.5, linewidth=1.5,
            color=colors[m], label=f'Network {m+1}' if m < 5 else None)
ax.scatter(X_train, y_train, c='black', s=20, zorder=5)
ax.plot(X_plot, y_true, 'k--', alpha=0.3)
ax.set_title(f'Deep Ensemble: {M} Independent Networks', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_ylim(-3, 3)
ax.legend(fontsize=8, ncol=2)

# Right: ensemble mean ± std
ax = axes[1]
ax.fill_between(X_plot.ravel(), ens_mu - 2*ens_std, ens_mu + 2*ens_std,
                alpha=0.15, color='#2ecc71', label='±2σ')
ax.fill_between(X_plot.ravel(), ens_mu - ens_std, ens_mu + ens_std,
                alpha=0.25, color='#2ecc71', label='±1σ')
ax.plot(X_plot, ens_mu, '#2ecc71', linewidth=2.5, label='Ensemble mean')
ax.plot(X_plot, y_true, 'k--', alpha=0.3, label='True function')
ax.scatter(X_train, y_train, c='black', s=20, zorder=5)
ax.set_title('Deep Ensemble: Mean ± Uncertainty Bands', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_ylim(-3, 3)
ax.legend(fontsize=10)

plt.suptitle('Approach 3: Deep Ensemble', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left: each colored line is an independently trained network.")
print("They agree where data is dense, disagree where it's sparse.")
print("\nRight: the disagreement becomes the uncertainty bands.")
print("This is the most similar to what Majdi showed with the noisy sine data.")

### Deep Ensemble Summary

| Pros | Cons |
|---|---|
| Best overall uncertainty estimates in practice | Must train M separate networks (M× training cost) |
| Captures both data noise AND model ignorance | M× memory to store all networks |
| Simple to implement (just train M networks) | Prediction requires M forward passes |
| No special loss function needed (can use NLL for even better results) | |

---

## Comparing All Three Approaches

In [ ]:
# ============================================================
# Side-by-side comparison
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

approaches = [
    ('MC Dropout', mc_mean, mc_std, 'steelblue'),
    ('Mean-Variance Network', mu_pred, sigma_pred, '#e74c3c'),
    ('Deep Ensemble', ens_mu, ens_std, '#2ecc71'),
]

for ax, (name, mu, sigma, color) in zip(axes, approaches):
    ax.fill_between(X_plot.ravel(), mu - 2*sigma, mu + 2*sigma,
                    alpha=0.15, color=color)
    ax.fill_between(X_plot.ravel(), mu - sigma, mu + sigma,
                    alpha=0.25, color=color)
    ax.plot(X_plot, mu, color=color, linewidth=2.5)
    ax.plot(X_plot, y_true, 'k--', alpha=0.3)
    ax.scatter(X_train, y_train, c='black', s=15, zorder=5)
    ax.set_title(name, fontweight='bold', fontsize=13, color=color)
    ax.set_xlabel('x')
    ax.set_ylim(-3, 3)

plt.suptitle('Three Approaches to Neural Network Uncertainty — Same Data, Different Methods',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Compare uncertainty in specific regions
# ============================================================
regions = [
    ('Dense data (x≈1.5)', 40),    # index in X_plot
    ('Sparse region (x≈5)', 120),
    ('Dense data (x≈8.5)', 190),
    ('Extrapolation (x≈11)', 240),
]

print("Uncertainty by Region and Method:")
print("=" * 65)
print(f"{'Region':<28s} {'MC Dropout':>10s} {'Mean-Var':>10s} {'Ensemble':>10s}")
print("-" * 65)
for name, idx in regions:
    print(f"{name:<28s} {mc_std[idx]:>10.4f} {sigma_pred[idx]:>10.4f} {ens_std[idx]:>10.4f}")
print("=" * 65)
print()
print("All methods should show: low uncertainty in dense regions,")
print("high uncertainty in sparse/extrapolation regions.")
print("The magnitudes differ because each method defines 'uncertainty' differently.")

---

## Two Types of Uncertainty

Michigan introduced these terms, and they're important for your projects:

**Aleatoric uncertainty** (data noise) — irreducible randomness in the data itself. Even with infinite data, your measurements have noise. A mean-variance network captures this through its predicted σ.

**Epistemic uncertainty** (model ignorance) — uncertainty due to not having enough data. If you collected more data in the sparse region, this uncertainty would shrink. MC Dropout and ensemble disagreement capture this.

Deep Ensembles with NLL loss capture **both** — the individual networks' σ captures aleatoric uncertainty, and the disagreement between networks captures epistemic uncertainty. This is one reason they're considered the gold standard.

In [ ]:
# ============================================================
# Visualize the two types of uncertainty
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Aleatoric: consistent noise everywhere (doesn't shrink with more models)
ax = axes[0]
np.random.seed(0)
x_demo = np.linspace(0, 6, 100)
y_demo = np.sin(x_demo)
for _ in range(20):
    noise = 0.5 * np.random.randn(100)  # same noise level everywhere
    ax.scatter(x_demo, y_demo + noise, alpha=0.05, s=10, color='#e74c3c')
ax.plot(x_demo, y_demo, 'k-', linewidth=2)
ax.set_title('Aleatoric Uncertainty\n(data noise — irreducible)', fontweight='bold', color='#e74c3c')
ax.set_xlabel('x')
ax.set_ylabel('y')

# Epistemic: high in gaps, low near data (shrinks with more data)
ax = axes[1]
ax.fill_between(X_plot.ravel(), ens_mu - 2*ens_std, ens_mu + 2*ens_std,
                alpha=0.3, color='#3498db')
ax.plot(X_plot, ens_mu, '#3498db', linewidth=2)
ax.scatter(X_train, y_train, c='black', s=20, zorder=5)
ax.set_title('Epistemic Uncertainty\n(model ignorance — reducible with more data)', 
             fontweight='bold', color='#3498db')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_ylim(-3, 3)

# Both together
ax = axes[2]
combined_std = np.sqrt(ens_std**2 + 0.3**2)  # approximate: ensemble + noise
ax.fill_between(X_plot.ravel(), ens_mu - 2*combined_std, ens_mu + 2*combined_std,
                alpha=0.15, color='#2ecc71', label='Total (aleatoric + epistemic)')
ax.fill_between(X_plot.ravel(), ens_mu - 2*ens_std, ens_mu + 2*ens_std,
                alpha=0.25, color='#3498db', label='Epistemic only')
ax.plot(X_plot, ens_mu, 'k', linewidth=2)
ax.scatter(X_train, y_train, c='black', s=20, zorder=5)
ax.set_title('Total Uncertainty\n(both types combined)', fontweight='bold', color='#2ecc71')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_ylim(-3, 3)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Aleatoric (left):  Same noise everywhere. You can't reduce it — it's in the data.")
print("Epistemic (middle): Large in gaps, small near data. Collect more data → it shrinks.")
print("Total (right):     What you actually care about in practice — both sources combined.")

---

## Summary: Connecting Everything

| Method | Training | Prediction | What It Captures |
|---|---|---|---|
| **Gaussian Process** | Bayesian (exact) | Mean + Std | Both, but doesn't scale |
| **MC Dropout** | Standard (with dropout) | N forward passes → mean, std | Primarily epistemic |
| **Mean-Variance Network** | Special NLL loss | One pass → μ, σ | Primarily aleatoric |
| **Deep Ensemble** | Train M networks | M passes → combined μ, σ | Both |

### For Your Projects

Michigan will assign each group one of these approaches. Regardless of which you get, the workflow is:

1. Train the model(s) on your data
2. Generate predictions with uncertainty (bands, error bars)
3. Check: is the uncertainty **calibrated**? (Do ~95% of true values fall within ±2σ?)
4. Check: is the uncertainty **useful**? (Is it larger where the model is actually wrong?)
5. Compare to a standard NN without uncertainty

The concepts from our probability notebook — sampling, mean, standard deviation, distributions, Bayesian updating — are the language you'll use to describe and evaluate your results.